# 01 — Exploratory data analysis and validation\n\nThis notebook performs data quality checks, distribution analysis, and establishes drift detection baselines for the MLOps pipeline. We validate the training data against the expected schema and inspect feature distributions that will later be monitored in production.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_DIR)

from src.data_loader import (
    load_training_data, load_drift_data, validate_schema,
    validate_nulls, validate_distributions, run_all_validations,
    EXPECTED_SCHEMA, EXPECTED_COLUMNS,
)

## Load datasets

In [ ]:
data_dir = os.path.join(PROJECT_DIR, "data")
train_df, test_df = load_training_data(data_dir)
drift_df = load_drift_data(data_dir)

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")
print(f"Drift: {drift_df.shape}")

In [ ]:
train_df.head()

In [ ]:
train_df.info()

## Schema validation\n\nVerify that columns, dtypes, and null rates match the expected schema defined in `data_loader.py`.

In [ ]:
schema_result = validate_schema(train_df)
null_result = validate_nulls(train_df)

print("Schema validation:", schema_result.summary())
print("Null validation:", null_result.summary())
print()

# Show any failures
for check in schema_result.to_dict() + null_result.to_dict():
    if not check["passed"]:
        print(f"  FAIL: {check['check']} — {check['detail']}")

In [ ]:
# Full validation report
full_result = run_all_validations(train_df)
val_df = pd.DataFrame(full_result.to_dict())
print(f"Overall: {full_result.summary()}")
val_df[val_df["passed"] == False]

## Target distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, df) in zip(axes, [("Train", train_df), ("Test", test_df), ("Drift", drift_df)]):
    counts = df["Churn"].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color=["#3B6FD4", "#E8C230"])
    ax.set_title(f"{name} — churn rate {df['Churn'].mean():.2%}")
    ax.set_xlabel("Churn")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Numeric feature distributions\n\nCompare training and drift data distributions side by side. The drift dataset has shifted distributions in tenure, MonthlyCharges, and contract mix.

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    ax.hist(train_df[col], bins=40, alpha=0.6, color="#3B6FD4", label="Train", density=True)
    ax.hist(drift_df[col], bins=40, alpha=0.6, color="#E8C230", label="Drift", density=True)
    ax.set_title(col)
    ax.legend()
plt.suptitle("Training vs drift distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics comparison
summary = pd.concat([
    train_df[numeric_cols].describe().T.add_prefix("train_"),
    drift_df[numeric_cols].describe().T.add_prefix("drift_"),
], axis=1)
summary[["train_mean", "drift_mean", "train_std", "drift_std"]]

## Categorical feature distributions

In [ ]:
cat_cols = ["Contract", "InternetService", "PaymentMethod"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, cat_cols):
    train_pct = train_df[col].value_counts(normalize=True).sort_index()
    drift_pct = drift_df[col].value_counts(normalize=True).sort_index()
    x = np.arange(len(train_pct))
    w = 0.35
    ax.bar(x - w/2, train_pct.values, w, label="Train", color="#3B6FD4")
    ax.bar(x + w/2, drift_pct.reindex(train_pct.index, fill_value=0).values, w, label="Drift", color="#E8C230")
    ax.set_xticks(x)
    ax.set_xticklabels(train_pct.index, rotation=30, ha="right", fontsize=8)
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Drift detection baseline (PSI)\n\nCompute PSI between training and drift data to establish the monitoring baseline. This will be used in production to decide when retraining is needed.

In [ ]:
from src.model import compute_psi

psi_scores = compute_psi(train_df, drift_df)
psi_df = pd.DataFrame([
    {"Feature": k, "PSI": round(v, 4),
     "Status": "Significant shift" if v > 0.25 else ("Moderate shift" if v > 0.10 else "No shift")}
    for k, v in sorted(psi_scores.items(), key=lambda x: -x[1])
])
psi_df

In [ ]:
colors = ["#ef4444" if v > 0.25 else "#f59e0b" if v > 0.10 else "#22c55e" for v in psi_df["PSI"]]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(psi_df["Feature"], psi_df["PSI"], color=colors)
ax.axvline(x=0.25, color="#ef4444", linestyle="--", label="Retrain threshold (0.25)")
ax.axvline(x=0.10, color="#f59e0b", linestyle=":", label="Investigation threshold (0.10)")
ax.set_xlabel("PSI")
ax.set_title("Population Stability Index — training vs drift data")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Correlation analysis

In [ ]:
corr_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "Churn"]
corr = train_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Correlation matrix — numeric features and target")
plt.tight_layout()
plt.show()

## Summary\n\n- All schema and null checks pass on the training data\n- The drift dataset shows clear shifts in tenure (lower), MonthlyCharges (higher), and contract mix (more month-to-month)\n- PSI scores confirm significant drift in multiple features, validating the monitoring approach\n- These baselines will be used by the pipeline to trigger automated retraining when PSI exceeds 0.25